In [34]:
import numpy as np 
import pandas as pd
from matplotlib import pyplot as plt 
from pathlib import Path

## First, we will convert american / decimal odds to implied probability, and then remove the vig added by the sportsbooks.

In [6]:
def odds_to_prob(odds, odds_type):
    odds = np.asarray(odds, dtype=float)
    res = np.zeros_like(odds, dtype = float)
    if odds_type == 'american':
    
        for i in range(len(odds)):
            if odds[i] > 0:
                res[i] = 100/(odds[i] +100)
            elif odds[i] <0 :
                res[i] = abs(odds[i])/(abs(odds[i]) + 100)
            else:
                raise ValueError('american odds cannot be 0')
    else:
        raise ValueError(f'unsupported odds_type entered: {odds_type}')
    return res

odds = np.array([600, -200, 325])
test_res = odds_to_prob(odds, odds_type = 'american')
print(test_res, sum(test_res))


def remove_vig(prob_arr):
    prob_arr = np.asarray(prob_arr, dtype=float)
    tot = np.sum(prob_arr)
    corr_arr = np.zeros_like(prob_arr, dtype=float)

    for i in range(len(prob_arr)):
        corr_arr[i] = prob_arr[i] / tot

    return corr_arr

test_novig = remove_vig(test_res)
print(f'sum is now = {sum(test_novig)}, and the values are {test_novig}')


[0.14285714 0.66666667 0.23529412] 1.0448179271708684
sum is now = 1.0, and the values are [0.13672922 0.63806971 0.22520107]


## These functions can only be applied to odds for the same game, same bet type, and from the same sportsbook, so we need to create a function that will apply this to a group formed from a pandas dataframe

In [7]:
def devig_group(group):
    raw_prob = odds_to_prob(group['american_odds'].values, odds_type = 'american')
    novig_prob = remove_vig(raw_prob)
    group = group.copy()
    group['raw_prob'] = raw_prob
    group['novig_prob'] = novig_prob
    return group

In [8]:
markets = pd.DataFrame({
    'match' : ['match1']*6,
    'question': ['game result']*6,
    'outcome': ['team A', 'team B', 'draw']*2,
    'book' : ['book 1']*3 + ['book 2']*3,
    'american_odds' : [600, -200, 325, 575, -190, 330],

})
    
markets
    

,match,question,outcome,book,american_odds
0,match1,game result,team A,book 1,600
1,match1,game result,team B,book 1,-200
2,match1,game result,draw,book 1,325
3,match1,game result,team A,book 2,575
4,match1,game result,team B,book 2,-190
5,match1,game result,draw,book 2,330


In [9]:
group_cols = ["match", "question", "book"]
markets['raw_prob'] = odds_to_prob(markets['american_odds'].values, odds_type='american')
markets['novig_prob']= markets.groupby(group_cols)['raw_prob'].transform(remove_vig)

markets

,match,question,outcome,book,american_odds,raw_prob,novig_prob
0,match1,game result,team A,book 1,600,0.142857,0.136729
1,match1,game result,team B,book 1,-200,0.666667,0.638070
2,match1,game result,draw,book 1,325,0.235294,0.225201
3,match1,game result,team A,book 2,575,0.148148,0.143017
4,match1,game result,team B,book 2,-190,0.655172,0.632480
5,match1,game result,draw,book 2,330,0.232558,0.224503


In [12]:
markets.groupby(group_cols, as_index=False)['novig_prob'].sum()

,match,question,book,novig_prob
0,match1,game result,book 1,1.0
1,match1,game result,book 2,1.0


In [20]:
consensus = markets.groupby(['match', 'question', 'outcome',], as_index = False)['novig_prob'].median().rename(columns = {"novig_prob":"market_prob"})

In [22]:
consensus['market_prob'].sum()


np.float64(1.0)

In [26]:
consensus = (
    markets
    .groupby(["match", "question", "outcome"], as_index=False)
    .agg(mean_prob=("novig_prob", "mean"), median_prob=("novig_prob", "median"), min_prob=("novig_prob", "min"), 
        max_prob=("novig_prob", "max"), 
        std_prob=("novig_prob", "std"),
        num_books=("book", "nunique")
    )
)

consensus["market_prob"] = consensus["mean_prob"]

consensus["submit_prob"] = (
    consensus["market_prob"]
    / consensus.groupby(["match", "question"])["market_prob"].transform("sum")
)

sums = consensus.groupby(["match", "question"])["submit_prob"].sum()
assert np.allclose(sums, 1.0), f"Normalization failed: {sums[~np.isclose(sums, 1.0)]}"
consensus

,match,question,outcome,mean_prob,median_prob,min_prob,max_prob,std_prob,num_books,market_prob,submit_prob
0,match1,game result,draw,0.224852,0.224852,0.224503,0.225201,0.000493,2,0.224852,0.224852
1,match1,game result,team A,0.139873,0.139873,0.136729,0.143017,0.004446,2,0.139873,0.139873
2,match1,game result,team B,0.635275,0.635275,0.632480,0.638070,0.003953,2,0.635275,0.635275


### Here, submit probability is the renormalized average across all sportsbooks considered

In [27]:
# Claude's helper function

def process_match(markets):
    """
    Takes a markets dataframe (raw odds from multiple books for one or more
    matches/questions) and returns a consensus dataframe with submit_prob
    plus diagnostic columns.

    Expected input columns: match, question, outcome, book, american_odds
    """
    df = markets.copy()  # never mutate the caller's dataframe

    # Step 1: raw implied probabilities
    df["raw_prob"] = odds_to_prob(df["american_odds"].values, odds_type="american")

    # Step 2: de-vig within each book's market
    group_cols = ["match", "question", "book"]
    df["novig_prob"] = (
        df.groupby(group_cols)["raw_prob"]
        .transform(remove_vig)
    )

    # Step 3: aggregate across books per outcome, with diagnostics
    consensus = (
        df.groupby(["match", "question", "outcome"], as_index=False)
        .agg(
            mean_prob=("novig_prob", "mean"),
            median_prob=("novig_prob", "median"),
            min_prob=("novig_prob", "min"),
            max_prob=("novig_prob", "max"),
            std_prob=("novig_prob", "std"),
            num_books=("book", "nunique"),
        )
    )

    # Step 4: pick the point estimate and normalize within each market
    consensus["market_prob"] = consensus["mean_prob"]
    consensus["submit_prob"] = (
        consensus["market_prob"]
        / consensus.groupby(["match", "question"])["market_prob"].transform("sum")
    )

    # Step 5: sanity check
    sums = consensus.groupby(["match", "question"])["submit_prob"].sum()
    assert np.allclose(sums, 1.0), f"Normalization failed:\n{sums[~np.isclose(sums, 1.0)]}"

    return consensus

In [28]:
process_match(markets)

,match,question,outcome,mean_prob,median_prob,min_prob,max_prob,std_prob,num_books,market_prob,submit_prob
0,match1,game result,draw,0.224852,0.224852,0.224503,0.225201,0.000493,2,0.224852,0.224852
1,match1,game result,team A,0.139873,0.139873,0.136729,0.143017,0.004446,2,0.139873,0.139873
2,match1,game result,team B,0.635275,0.635275,0.632480,0.638070,0.003953,2,0.635275,0.635275


In [29]:
# Chat GPT version

def process_match(markets, normalize=False):
    """
    Takes a markets dataframe and returns:
      df: detailed book-level table with raw_prob and novig_prob
      consensus: aggregated probabilities to submit
      submit_sums: sanity-check table

    Expected input columns:
      forecast_run_id, match_date, match, question, outcome, book, american_odds, notes
    """

    required_cols = {
        "forecast_run_id",
        "match_date",
        "match",
        "question",
        "outcome",
        "book",
        "american_odds",
        "notes",
    }

    missing = required_cols - set(markets.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = markets.copy()

    # Step 1: raw implied probabilities
    df["raw_prob"] = odds_to_prob(
        df["american_odds"].values,
        odds_type="american"
    )

    # Step 2: de-vig within each book's version of the market
    book_market_cols = [
        "forecast_run_id",
        "match",
        "question",
        "book",
    ]

    df["novig_prob"] = (
        df
        .groupby(book_market_cols)["raw_prob"]
        .transform(remove_vig)
    )

    # Step 3: sanity check book-level probabilities
    book_sums = (
        df
        .groupby(book_market_cols, as_index=False)["novig_prob"]
        .sum()
        .rename(columns={"novig_prob": "book_novig_sum"})
    )

    bad_book_sums = book_sums[
        ~np.isclose(book_sums["book_novig_sum"], 1.0)
    ]

    if len(bad_book_sums) > 0:
        raise ValueError(
            f"Book-level no-vig probabilities do not sum to 1:\n{bad_book_sums}"
        )

    # Step 4: aggregate across books
    consensus = (
        df
        .groupby(
            [
                "forecast_run_id",
                "match_date",
                "match",
                "question",
                "outcome",
            ],
            as_index=False
        )
        .agg(
            mean_prob=("novig_prob", "mean"),
            median_prob=("novig_prob", "median"),
            min_prob=("novig_prob", "min"),
            max_prob=("novig_prob", "max"),
            std_prob=("novig_prob", "std"),
            num_books=("book", "nunique"),
        )
    )

    consensus["market_prob"] = consensus["mean_prob"]

    # Step 5: check market-level sums before normalization
    market_cols = ["forecast_run_id", "match", "question"]

    consensus["market_prob_sum"] = (
        consensus
        .groupby(market_cols)["market_prob"]
        .transform("sum")
    )

    bad_market_sums = consensus[
        ~np.isclose(consensus["market_prob_sum"], 1.0)
    ]

    if len(bad_market_sums) > 0:
        print("Warning: market probabilities do not sum to 1 before normalization.")
        print(
            bad_market_sums[
                market_cols + ["outcome", "market_prob", "market_prob_sum"]
            ]
        )

    # Step 6: final submission probabilities
    if normalize:
        consensus["submit_prob"] = (
            consensus["market_prob"] / consensus["market_prob_sum"]
        )
    else:
        consensus["submit_prob"] = consensus["market_prob"]

    submit_sums = (
        consensus
        .groupby(market_cols, as_index=False)["submit_prob"]
        .sum()
        .rename(columns={"submit_prob": "submit_prob_sum"})
    )

    bad_submit_sums = submit_sums[
        ~np.isclose(submit_sums["submit_prob_sum"], 1.0)
    ]

    if len(bad_submit_sums) > 0:
        raise ValueError(
            f"Final submit probabilities do not sum to 1:\n{bad_submit_sums}"
        )

    return df, consensus, submit_sums

In [30]:
markets = pd.DataFrame({
    "forecast_run_id": ["2026-06-20_match1_test"] * 6,
    "match_date": ["2026-06-20"] * 6,
    "match": ["match1"] * 6,
    "question": ["game result"] * 6,
    "outcome": ["team A", "team B", "draw"] * 2,
    "book": ["book 1"] * 3 + ["book 2"] * 3,
    "american_odds": [600, -200, 325, 575, -190, 330],
    "notes": [""] * 6,
})

In [31]:
df, consensus, submit_sums = process_match(markets)

df

,forecast_run_id,match_date,match,question,outcome,book,american_odds,notes,raw_prob,novig_prob
0,2026-06-20_match1_test,2026-06-20,match1,game result,team A,book 1,600,,0.142857,0.136729
1,2026-06-20_match1_test,2026-06-20,match1,game result,team B,book 1,-200,,0.666667,0.638070
2,2026-06-20_match1_test,2026-06-20,match1,game result,draw,book 1,325,,0.235294,0.225201
3,2026-06-20_match1_test,2026-06-20,match1,game result,team A,book 2,575,,0.148148,0.143017
4,2026-06-20_match1_test,2026-06-20,match1,game result,team B,book 2,-190,,0.655172,0.632480
5,2026-06-20_match1_test,2026-06-20,match1,game result,draw,book 2,330,,0.232558,0.224503


In [32]:
consensus

,forecast_run_id,match_date,match,question,outcome,mean_prob,median_prob,min_prob,max_prob,std_prob,num_books,market_prob,market_prob_sum,submit_prob
0,2026-06-20_match1_test,2026-06-20,match1,game result,draw,0.224852,0.224852,0.224503,0.225201,0.000493,2,0.224852,1.0,0.224852
1,2026-06-20_match1_test,2026-06-20,match1,game result,team A,0.139873,0.139873,0.136729,0.143017,0.004446,2,0.139873,1.0,0.139873
2,2026-06-20_match1_test,2026-06-20,match1,game result,team B,0.635275,0.635275,0.632480,0.638070,0.003953,2,0.635275,1.0,0.635275


In [33]:
submit_sums

,forecast_run_id,match,question,submit_prob_sum
0,2026-06-20_match1_test,match1,game result,1.0


### Now save the data

In [37]:
def append_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        df.to_csv(path, mode="a", header=False, index=False)
    else:
        df.to_csv(path, index=False)

In [38]:
new_odds = pd.DataFrame({
    "forecast_run_id": ["2026-06-20_some_match_pregame"] * 6,
    "match_date": ["2026-06-20"] * 6,
    "match": ["Team A vs Team B"] * 6,
    "question": ["Match result"] * 6,
    "outcome": ["Team A", "Team B", "Draw"] * 2,
    "book": ["Pinnacle"] * 3 + ["DraftKings"] * 3,
    "american_odds": [150, 180, 240, 145, 185, 245],
    "notes": [""] * 6,
})

df, consensus, submit_sums = process_match(new_odds)

consensus[["outcome", "submit_prob", "mean_prob", "median_prob", "min_prob", "max_prob", "std_prob", "num_books"]]

,outcome,submit_prob,mean_prob,median_prob,min_prob,max_prob,std_prob,num_books
0,Draw,0.278060,0.278060,0.278060,0.276343,0.279776,0.002428,2
1,Team A,0.384816,0.384816,0.384816,0.380496,0.389136,0.006110,2
2,Team B,0.337124,0.337124,0.337124,0.334521,0.339728,0.003682,2


In [39]:
odds_to_save = new_odds.copy()
odds_to_save["entered_at"] = pd.Timestamp.now(tz="UTC")

append_csv(odds_to_save, "data/odds_log.csv")